In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 12. 토크나이저 — 텍스트를 숫자로 ⭐

> **학습 목표**
> - 서브워드 토큰화가 왜 LLM의 표준인지 이해한다
> - BPE 알고리즘을 처음부터 Python으로 구현한다
> - WordPiece, Unigram Language Model의 차이를 설명한다
> - 바이트 단위 토크나이저(Byte-level BPE)의 장점을 안다

## 12.1 토큰화의 어려움

세 가지 극단:
- **단어 단위**: "Hello, world!" → ["Hello", ",", "world", "!"]
  - 문제: 어휘가 무한대, 오타/신조어 대응 불가, OOV(Out-Of-Vocabulary)
- **문자 단위**: "Hello" → ['H', 'e', 'l', 'l', 'o']
  - 문제: 시퀀스 너무 김, 의미 단위 상실
- **서브워드 단위**: "unhappiness" → ["un", "happiness"] 또는 ["un", "happ", "iness"]
  - **LLM 표준**: 빈도 높은 건 통째로, 낮은 건 쪼개서

GPT-2: Byte-level BPE. BERT: WordPiece. T5: Unigram.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# 단어 단위 토큰화 예시
text = "The quick brown fox jumps over the lazy dog."
word_tokens = text.lower().replace('.', ' .').split()
print(f"word 단위: {word_tokens}")

# Character-level
char_tokens = list(text.lower())
print(f"Character-level: {char_tokens}")

# 서브워드 (Concept적 예시)
subword_tokens = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
print(f"서브워드 단위 (예시): {subword_tokens}")


## 12.2 Byte-Pair Encoding (BPE) — GPT-2의 선택

**알고리즘**:
1. 모든 단어를 문자 시퀀스로 초기화
2. 가장 빈도가 높은 인접 쌍(2-gram)을 병합
3. 어휘 크기에 도달할 때까지 반복

예: 'low' (5회), 'lower' (2회), 'newest' (6회), 'widest' (3회)

빈도 테이블에서 'e'+'s' 쌍이 9회 → 병합 → 'es'


In [ ]:
# BPE 직접 구현
from collections import Counter

class BPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []  # (a, b) 순서대로 병합 규칙
        self.vocab = set()

    def _get_word_freqs(self, corpus):
        """word 빈도 Computation."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for w in words:
                word_freqs[w] += 1
        return word_freqs

    def _get_pair_freqs(self, splits, word_freqs):
        """인접 쌍 빈도 Computation."""
        pair_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
        return pair_freqs

    def _merge(self, pair, splits):
        """모든 word에서 pair를 병Sum."""
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = self._get_word_freqs(corpus)
        # 각 word를 Character-level로 Decomposition (끝에 </w> Display)
        splits = {w: list(w) + ['</w>'] for w in word_freqs}
        # 초기 어휘
        self.vocab = set(c for w in word_freqs for c in w) | {'</w>'}

        for _ in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(splits, word_freqs)
            if not pair_freqs:
                break
            best_pair = max(pair_freqs, key=pair_freqs.get)
            splits = self._merge(best_pair, splits)
            self.merges.append(best_pair)
            self.vocab.add(best_pair[0] + best_pair[1])

    def encode(self, word):
        """Training된 규칙으로 word 인코딩."""
        split = list(word) + ['</w>']
        for a, b in self.merges:
            i = 0
            new_split = []
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            split = new_split
        return split

# 작은 코퍼스로 Training
corpus = [
    "low low low low low",
    "lower lower newest newest newest",
    "widest widest newest newest",
]
bpe = BPE(num_merges=10)
bpe.train(corpus)

print("Training된 병Sum 규칙:")
for i, (a, b) in enumerate(bpe.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")
print(f"\nVocabulary Size: {len(bpe.vocab)}")

# 인코딩 Test
for word in ['low', 'lowest', 'newer', 'widest']:
    tokens = bpe.encode(word)
    print(f"  {word:>10} -> {tokens}")


## 12.3 WordPiece — BERT의 선택

BPE와 비슷하지만 **병합 기준이 다름**:

$$\text{score}(a, b) = \frac{\text{freq}(ab)}{\text{freq}(a) \cdot \text{freq}(b)}$$

BPE는 빈도 자체를 최대화, WordPiece는 빈도를 정규화하여 점수 최대화.
이는 자주 등장하지만 우연히 같이 나오는 쌍보다, 진짜로 함께 쓰이는 쌍을 선호.


In [ ]:
# WordPiece 간단 구현
class WordPiece:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []

    def _get_pair_scores(self, splits, word_freqs):
        pair_freqs = Counter()
        char_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
            for s in split:
                char_freqs[s] += word_freqs[word]
        scores = {p: f / (char_freqs[p[0]] * char_freqs[p[1]]) for p, f in pair_freqs.items()}
        return scores

    def _merge(self, pair, splits):
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        # WordPiece는 ## 접두사로 단어 내부 토큰 표시
        splits = {}
        for w in word_freqs:
            chars = list(w)
            splits[w] = [chars[0]] + ['##' + c for c in chars[1:]]
        for _ in range(self.num_merges):
            scores = self._get_pair_scores(splits, word_freqs)
            if not scores:
                break
            best = max(scores, key=scores.get)
            splits = self._merge(best, splits)
            self.merges.append(best)

wp = WordPiece(num_merges=10)
wp.train(corpus)
print("WordPiece Training된 병Sum:")
for i, (a, b) in enumerate(wp.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")


## 12.4 Unigram Language Model — T5의 선택

BPE/WordPiece는 **bottom-up** (문자에서 병합). Unigram은 **top-down** (큰 어휘에서 삭제).

1. 초기 어휘 = 모든 서브워드 후보 (큼)
2. 각 서브워드 확률 $P(t) = \text{count}(t) / \sum \text{count}$
3. 손실 $\mathcal{L} = -\sum_i \log P(\mathbf{x}_i) = -\sum_i \log \sum_{\text{seg}} \prod P(t)$
4. 손실 증가가 가장 작은 토큰을 삭제
5. 반복

EM 알고리즘으로 최적 분할 찾음.


In [ ]:
# Unigram 간단 구현 (개념적)
from itertools import combinations

class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}  # token -> prob

    def _get_candidates(self, word_freqs, max_len=10):
        candidates = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word)):
                for j in range(i+1, min(i+max_len+1, len(word)+1)):
                    candidates[word[i:j]] += freq
        return candidates

    def _segment(self, word, vocab):
        """Viterbi로 Optimal 분할."""
        n = len(word)
        # dp[i] = (best_log_prob, best_segmentation)
        dp = [(-float('inf'), None)] * (n + 1)
        dp[0] = (0, [])
        for i in range(1, n + 1):
            for j in range(max(0, i - 10), i):
                token = word[j:i]
                if token in vocab:
                    log_p = np.log(vocab[token])
                    if dp[j][0] + log_p > dp[i][0]:
                        dp[i] = (dp[j][0] + log_p, dp[j][1] + [token])
        return dp[n]

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        candidates = self._get_candidates(word_freqs)
        total = sum(candidates.values())
        self.vocab = {t: c / total for t, c in candidates.items()}

        # Iteration적으로 Loss Increase가 작은 토큰 삭제
        while len(self.vocab) > self.vocab_size:
            # 각 토큰 삭제 시 Loss Increase Computation
            losses = {}
            for token in list(self.vocab.keys()):
                if len(token) == 1:  # 한 글자는 유지
                    continue
                # 토큰 삭제했을 때의 Loss (간소화)
                losses[token] = self.vocab[token]  # 간단히 Probability을 loss로
            if not losses:
                break
            # Loss이 가장 작은 토큰들 삭제
            sorted_tokens = sorted(losses.items(), key=lambda x: x[1])
            n_remove = max(1, len(self.vocab) - self.vocab_size)
            for token, _ in sorted_tokens[:n_remove]:
                del self.vocab[token]

# 작은 예시
uni = UnigramTokenizer(vocab_size=30)
uni.train(corpus)
print(f"Unigram Vocabulary Size: {len(uni.vocab)}")
print(f"상위 토큰: {sorted(uni.vocab.items(), key=lambda x: -x[1])[:10]}")

# 분할 예시
for word in ['lowest', 'newest']:
    log_p, seg = uni._segment(word, uni.vocab)
    print(f"  {word} -> {seg} (log_p={log_p:.4f})")


## 12.5 Byte-level BPE — GPT-2/3의 선택

문제: 유니코드 문자가 너무 많음 (한글 11,172자, 이모지 등).
해결: 모든 텍스트를 **UTF-8 바이트**로 인코딩 후 BPE 적용.

- 바이트는 256개뿐 → 기본 어휘 256
- UNK(unknown) 토큰 불필요 — 모든 문자열 표현 가능
- 다국어 처리 용이

GPT-2 어휘: 256 바이트 + 50,257 병합 ≈ 50K 토큰


In [ ]:
# Byte-level BPE 시연
text = "Hello, 안녕하세요!"
bytes_seq = text.encode('utf-8')
print(f"원본: {text}")
print(f"UTF-8 바이트: {list(bytes_seq)}")
print(f"바이트 수: {len(bytes_seq)} (문자 수 {len(text)})")
print("\n=> 모든 문자를 바이트로 표현 가능. UNK 불필요.")

# HuggingFace tokenizers 라이브러리 사용
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
except ImportError:
    print("\n[HuggingFace tokenizers 패키지 필요: pip install tokenizers]")

# 간단한 BPE Tokenizer (HuggingFace)
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])

    # 작은 코퍼스 Training
    from llm_math.data import load_tiny_shakespeare
    train_text = load_tiny_shakespeare(max_chars=5000)
    tokenizer.train_from_iterator([train_text], trainer)

    print(f"\nTraining된 Vocabulary Size: {tokenizer.get_vocab_size()}")
    # 인코딩 Test
    enc = tokenizer.encode("To be or not to be")
    print(f"'To be or not to be' -> {enc.tokens}")
    print(f"IDs: {enc.ids}")
except ImportError:
    print("\n[tokenizers 라이브러리 없음. pip install tokenizers로 설치하세요]")


## 12.6 [CPU/GPU 벤치마크 ④] 토크나이저 학습/인코딩 속도

토크나이저 작업은 CPU에서 수행되지만, 어휘 크기와 알고리즘에 따라 속도 차이가 크다.


In [ ]:
# 토크나이저 벤치마크 (CPU만)
import time
from llm_math.data import load_tiny_shakespeare

# 큰 코퍼스 준비
text = load_tiny_shakespeare(max_chars=100000)
print(f"벤치마크용 텍스트: {len(text):,}자")

# 1. 우리 BPE로 학습 시간 측정
def bench_bpe_train(text, n_merges, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        bpe = BPE(num_merges=n_merges)
        bpe.train([text])
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

print(f"\n{'merges':>8} | {'train (s)':>12}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    m, s = bench_bpe_train(text, n, repeat=2)
    print(f"{n:>8} | {m:>10.4f}±{s:.4f}")

# 2. 인코딩 Speed
bpe_small = BPE(num_merges=50)
bpe_small.train([text[:5000]])

def bench_encode(bpe, text, n_words=1000):
    words = text.split()[:n_words]
    t0 = time.perf_counter()
    for w in words:
        bpe.encode(w)
    return (time.perf_counter() - t0) * 1000

t_ms = bench_encode(bpe_small, text, n_words=1000)
print(f"\n1000단어 인코딩: {t_ms:.2f}ms ({t_ms/1000:.3f}ms/단어)")
print("\n=> Tokenizer는 CPU 작업. 어휘가 크면 검색이 느려진다.")
print("   HuggingFace tokenizers는 Rust 구현이라 훨씬 빠르다.")


## 12.7 어휘 크기와 성능 트레이드오프

- 어휘 작게 → 시퀀스 길어짐, 모델 느려짐, 임베딩 테이블 작음
- 어휘 크게 → 시퀀스 짧아짐, 임베딩 테이블 큼, 희귀 토큰 학습 부족

LLM별 어휘 크기:
- GPT-2: 50,257
- GPT-3: 50,257
- LLaMA-2: 32,000
- LLaMA-3: 128,256
- GPT-4: ~100,000


In [ ]:
# 어휘 크기별 시퀀스 길이 비교 (시뮬레이션)
# 작은 어휘는 더 많은 토큰으로 분할
np.random.seed(0)
text_sample = load_tiny_shakespeare(max_chars=5000)

# 다양한 어휘 크기로 학습하고 평균 토큰 수 비교
vocab_sizes = [30, 50, 100, 200, 500]
avg_tokens = []
for vs in vocab_sizes:
    bpe = BPE(num_merges=vs - 30)  # 초기 어휘 약 30
    bpe.train([text_sample])
    # 샘플 단어들의 평균 토큰 수
    sample_words = text_sample.split()[:100]
    total_tokens = sum(len(bpe.encode(w)) for w in sample_words)
    avg_tokens.append(total_tokens / len(sample_words))

plt.figure(figsize=(9, 5))
plt.plot(vocab_sizes, avg_tokens, 'o-', linewidth=2, markersize=8)
plt.xlabel('Vocabulary Size')
plt.ylabel('Mean 토큰 수 / word')
plt.title('Vocabulary Size와 시퀀스 길이의 Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch12_vocab_size_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> 어휘가 커질수록 word당 토큰 수가 줄어든다. 하지만 Embedding 테이블은 커진다.")


## 12.8 핵심 정리

| 알고리즘 | 병합 기준 | 사용 모델 |
|---|---|---|
| BPE | 빈도 최대 | GPT-2 |
| WordPiece | $\frac{\text{freq}(ab)}{\text{freq}(a)\text{freq}(b)}$ | BERT |
| Unigram | 손실 최소 (top-down) | T5 |
| Byte-level BPE | BPE on bytes | GPT-2/3/4 |

## 연습문제

1. 직접 구현한 BPE를 한국어 텍스트에 적용하고, 어려운 점을 기술하라.
2. BPE와 WordPiece를 같은 코퍼스에서 학습하고 병합 순서를 비교하라.
3. 어휘 크기 100, 500, 1000에 대해 평균 토큰 수를 비교하라.
4. Byte-level BPE가 왜 UNK 토큰을 없애는지 설명하라.
5. HuggingFace `tokenizers` 라이브러리로 GPT-2 토크나이저를 로드하고 한글 텍스트를 인코딩하라.

> 해설: `solutions/ch12_solutions.ipynb`
